# 03 — Treino e avaliação do detector YOLO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/projeto-plumas/blob/main/03_yolo_treino/03_yolo_treino.ipynb)

**Objetivo:** treinar o detector da classe **pluma** e comparar variantes de treino no mesmo conjunto de teste.

**Escopo da professora:** seção 6 (registrar versão, resolução, épocas, sementes, augmentations; examinar anotações vs. predições) e pergunta 1.

**Lê:** `02_anotacao_split/dataset.yaml` (e `dataset_B_publico.yaml`).

**Grava:** `runs/<variante>/weights/best.pt`, `metricas_deteccao.csv`, figuras de predições.

**GPU:** sim (Ambiente de execução → Alterar tipo → GPU T4)


In [ ]:
# === Preparo (igual em todos os notebooks) ===
import os, sys, subprocess

REPO = "USUARIO/projeto-plumas"      # troque pelo seu (veja README)
REPO_DIR = "/content/projeto-plumas"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    try:                                   # repositório privado: secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        _tok = None
    _url = f"https://{_tok}@github.com/{REPO}.git" if _tok else f"https://github.com/{REPO}.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)
    else:
        _r = subprocess.run(["git", "clone", "-q", _url, REPO_DIR], capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("Falha no clone. Repositório privado? Configure o secret GITHUB_TOKEN (veja README).")
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:                                      # execução local: sobe até a raiz do repo
    _p = os.getcwd()
    while not os.path.exists(os.path.join(_p, "config.yaml")) and os.path.dirname(_p) != _p:
        _p = os.path.dirname(_p)
    os.chdir(_p)
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from pathlib import Path
from plumas.config import load_config, stage_dir, set_seed, log_experiment
cfg = load_config()
set_seed(cfg["seed"])
STAGE = "03_yolo_treino"
D = stage_dir(cfg, STAGE)                  # pasta desta etapa no Drive
print("Etapa:", STAGE, "| dados em:", D)


## Plano de variantes (mesmo teste para todas)
| Variante | O que é |
|---|---|
| **A** | Zero-shot: pesos de fumaça já treinados por terceiros, **sem** treino nosso (linha de base) |
| **B** | Ajuste fino só com dados públicos |
| **C** | Ajuste fino com dados públicos + quadros próprios (treino) |

Troque `VARIANTE` e rode o treino para cada uma. A tabela `metricas_deteccao.csv` acumula as linhas.
Métricas: mAP50, mAP50-95, precisão e recall no **teste**.

In [ ]:
import torch, pandas as pd
print("GPU disponível:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
import ultralytics; print("ultralytics", ultralytics.__version__)

In [ ]:
from ultralytics import YOLO

VARIANTE = "C"          # "B" ou "C" (A é avaliada mais abaixo, sem treino)
y = cfg["yolo"]
D2 = stage_dir(cfg, "02_anotacao_split")
DATA = str(D2 / ("dataset_B_publico.yaml" if VARIANTE == "B" else "dataset.yaml"))
RUN_NAME = f"{VARIANTE}_{y['model'].replace('.pt', '')}_img{y['imgsz']}"

model = YOLO(y["model"])                         # pesos pré-treinados (COCO)
model.train(
    data=DATA, imgsz=y["imgsz"], epochs=y["epochs"], patience=y["patience"], batch=y["batch"],
    seed=cfg["seed"], deterministic=True,
    project=str(D / "runs"), name=RUN_NAME,      # saídas direto no Drive (a sessão do Colab pode cair)
    **y["augment"],
)
best = Path(model.trainer.best)
print("Melhores pesos:", best)

In [ ]:
def avaliar(weights, data, imgsz, nome, split="test"):
    m = YOLO(str(weights))
    r = m.val(data=data, split=split, imgsz=imgsz, conf=0.001, iou=0.6,
              project=str(D / "val"), name=nome, plots=True, verbose=False)
    row = {"variante": nome, "split": split, "imgsz": imgsz,
           "precision": float(r.box.mp), "recall": float(r.box.mr),
           "mAP50": float(r.box.map50), "mAP50_95": float(r.box.map),
           "pesos": str(weights)}
    csv = D / "metricas_deteccao.csv"
    old = pd.read_csv(csv) if csv.exists() else pd.DataFrame()
    pd.concat([old, pd.DataFrame([row])], ignore_index=True).to_csv(csv, index=False)
    return row

row = avaliar(best, DATA, y["imgsz"], RUN_NAME)
log_experiment(cfg, STAGE, {"variante": VARIANTE, "modelo": y["model"], "imgsz": y["imgsz"],
                            "epochs": y["epochs"], "seed": cfg["seed"], "augment": y["augment"],
                            "ultralytics": ultralytics.__version__, "metricas_teste": row})
pd.read_csv(D / "metricas_deteccao.csv")

## Variante A: zero-shot (linha de base)
Informe os pesos de um modelo de fumaça de terceiros (do GitHub/Hugging Face/Roboflow). **Verifique a licença** e que a classe de fumaça é a classe 0 do modelo; se não for, a avaliação não vale.

In [ ]:
ZERO_SHOT_WEIGHTS = None     # ex.: "/content/drive/MyDrive/modelos/fumaca_terceiros.pt"
if ZERO_SHOT_WEIGHTS:
    avaliar(ZERO_SHOT_WEIGHTS, str(D2 / "dataset.yaml"), y["imgsz"], "A_zero_shot")
pd.read_csv(D / "metricas_deteccao.csv")

## Examinar anotações vs. predições (exigido pelo escopo)
Verde = anotação manual, vermelho = predição. Olhe também os **erros** (nuvem/neblina confundidas com pluma, pluma não detectada, caixa grande demais) e escreva o que viu na célula de comentários abaixo.

In [ ]:
import cv2, matplotlib.pyplot as plt

def desenhar(img_path, lab_path, pred_boxes):
    img = cv2.imread(str(img_path)); h, w = img.shape[:2]
    if Path(lab_path).exists():
        for line in Path(lab_path).read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                cx, cy, bw, bh = map(float, p[1:]);
                cv2.rectangle(img, (int((cx-bw/2)*w), int((cy-bh/2)*h)), (int((cx+bw/2)*w), int((cy+bh/2)*h)), (0, 200, 0), 3)
    for (x1, y1, x2, y2, cf) in pred_boxes:
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 3)
        cv2.putText(img, f"{cf:.2f}", (int(x1), max(15, int(y1)-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

ds = D2 / "yolo_dataset"
test_imgs = sorted((ds / "images" / "test").glob("*.*"))[:: max(1, len(list((ds / "images" / "test").glob("*.*"))) // 8)][:8]
best_model = YOLO(str(best))
fig, axs = plt.subplots(2, 4, figsize=(20, 9))
for ax, p in zip(axs.ravel(), test_imgs):
    r = best_model.predict(str(p), conf=y["conf"], iou=y["iou"], verbose=False)[0]
    boxes = [(*b[:4], c) for b, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist())]
    ax.imshow(desenhar(p, ds / "labels" / "test" / (p.stem + ".txt"), boxes)); ax.set_title(p.name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

### Observações dos erros (preencher)
- Falsos positivos (o que o modelo confundiu com pluma): …
- Falsos negativos (pluma que ele perdeu, e em que condição): …
- Caixas imprecisas: …